# Public charging points in Dutch municipalities: need or affluence?

**TIL6022 Python Programming – Group 15**

Members: Merel Vos, Mirte Batema, Famke Lemstra, Luca Preijde, Mats de Vries

Repository: [github.com/mirtebatema/group-15](https://github.com/mirtebatema/group-15)

In [1]:
# Setup: the tables in this report are read from the results of notebooks 6 to 8
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd

PROCESSED = "data/processed/"


def stars(p):
    # Significance stars: * p < 0.05, ** p < 0.01, *** p < 0.001
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return ""


def signed(x, decimals=2):
    # Number with a + or - sign, rounded half up (0.305 becomes +0.31)
    rounded = Decimal(str(x)).quantize(Decimal(1).scaleb(-decimals), rounding=ROUND_HALF_UP)
    return f"{rounded:+}"


def p_text(p):
    # p-value as text, with '< 0.001' for very small values
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

## 1. Introduction and motivation

The Netherlands aims for a fast transition to electric passenger cars. To support this, The National Charging Infrastructure Agenda (NAL), part of the national Climate Agreement, sets the ambition of adding just over 300,000 public and 500,000 semi-public charging points by 2030 (Binnenlands Bestuur, 2024; VNG, 2026). Public charging points are essential for a large part of the population. Only 27% of Dutch households can park on their own property, compared to 67% of current EV drivers (RVO et al., 2025).

Municipalities play a key role in the development of this network. They manage public space and are responsible for planning, permits and the direction of the rollout (VNG, 2026). However, substantial differences exist between municipalities. In the data used in this study, the density of regular public charging points ranges from 0.5 to 42.9 per 1,000 households. International research shows that higher-income areas often have better access to public charging (Chen et al., 2025). Whether similar patterns exist across Dutch municipalities remains an important question.

The central question is therefore whether the distribution of public charging infrastructure reflects local need, such as high population density and limited access to private parking, or socioeconomic characteristics, such as income and education levels. Using open data for all 342 Dutch municipalities, this study examines how demographic, socioeconomic and political characteristics are associated with public charging point density. Understanding these relationships can help assess whether the rollout reflects local needs or whether certain municipalities risk falling behind in the transition to electric mobility.

## 2. Research objective and questions

**Objectives**

1. Map the variation in public charging point density across Dutch municipalities.
2. Examine how socioeconomic, demographic, and political characteristics relate to that density.
3. Determine with a multivariate regression which characteristics explain the most variation when taken together.
4. Assess whether the rollout is linked mainly to need (population density, share of apartments) or to socioeconomic characteristics (income, education).

**Main research question**

To what extent do socioeconomic, demographic and political characteristics explain differences in the number of public charging points per 1,000 households between Dutch municipalities, and which characteristics have the greatest explanatory power?

*Table 1. Sub-questions and expected relations.*

| Sub-question | Expected relation (hypothesis) |
| --- | --- |
| SQ1. How does charging point density vary between municipalities? | Large differences between municipalities. |
| SQ2. How is charging point density related to income and education level? | Positive. More EV owners in richer municipalities. |
| SQ3. How is charging point density related to age structure? | Positive for a young population, negative for a large 65+ share. |
| SQ4. How is charging point density related to political preference? | Positive for a higher share of left-wing or climate-oriented votes. |
| SQ5. How is charging point density related to population density and housing type? | Positive. Dense municipalities have fewer private driveways, so more need for public charging. |

## 3. Contribution statement

As a group we defined the research question, the variables and the scope of the study. Everyone contributed to the code and the main responsibilities per person are described below.

**Mirte Batema**
- Coding: cleaning the raw datasets and preparing them for the analysis (notebooks 0), map of income (notebook 1.2) and tables for the main report
- Reporting: Analysis and results (sections 6.4–6.5) and AI statement

**Merel Vos**
- Coding: maps of charging point density, population density, age and education (notebooks 1, 2, 2.2, 3 and 5), boxplot and scatter plots (notebook 6.2) and robustness checks of the regression (notebook 8)
- Reporting: Data used (chapter 4) and Data pipeline (chapter 5)

**Famke Lemstra**
- Coding: map of left-wing vote share (notebook 4), README file
- Reporting: Contribution statement (chapter 3) and Analysis and results (sections 6.1–6.3)

**Luca Preijde**
- Coding: descriptive statistics and comparison of all factors (notebook 6, section 6.6); multivariate regression and model diagnostics (notebook 7)
- Reporting: Introduction (chapter 1), Research objective and questions (chapter 2), Conclusion (chapter 8) and Discussion (chapter 9)

**Mats de Vries**
- Coding: multivariate regression and model diagnostics (notebook 7), conversion of the report into the final notebook (main report)
- Reporting: Regression model (chapter 7) and References (chapter 10)

## 4. Data used

The unit of analysis is the municipality (342 municipalities). The study is cross-sectional: each variable comes from a single observation period, although these periods differ between 2024 and 2026 (see Table 2). The dependent variable is the number of regular public charging points per 1,000 households. Households are used instead of inhabitants because charging demand is more closely related to households and their car ownership than to the number of individual residents. This also makes comparisons between municipalities with different average household sizes more accurate. Table 2 provides an overview of the datasets (the raw data) that are used in data preparation before the analyses could be carried out.

*Table 2. Overview of the raw datasets in data/raw/.*

| File in data/raw/ | Content | Period | Source |
| --- | --- | --- | --- |
| laadpunten_per_gemeente.csv | Regular public charging points per municipality | July 2026 | NAL (2026) |
| inkomens_per_gemeente.csv | Private households and mean and median standardised income per municipality | 2024 (provisional) | CBS (2025a) |
| jongeren_per_gemeente_2026.csv, ouderen_per_gemeente_2026.csv | Share of residents aged 0–25 and 65+ per municipality | 2026 | CBS (2026) |
| tk2025_uitslag.csv | Results of the Tweede Kamer election of 29 October 2025 per municipality and party | 2025 | Kiesraad (2025) |
| kwb2024.zip | Full download of CBS Kerncijfers wijken en buurten 2024, with all districts and neighbourhoods | 2024 | CBS (2025b) |
| gemeenten_2024.gpkg | Municipal boundaries (layer gemeente_gegeneraliseerd), retrieved from the PDOK web service | 2024 | CBS (2024) |


The datasets are joined on municipality name. The sources write some names differently. CBS writes *'s-Gravenhage (gemeente)* instead of *Den Haag* and *Beek (L.)* instead of *Beek*, and the Kiesraad writes *Hengelo (O)* and *Bergen (NH)* without the final full stop. These names are mapped by hand to the spelling of the charging point file before merging: nine names in the CBS files and four in the election results. After this step all 342 municipalities match in every dataset. For the maps of education level and population density, the data is joined on the CBS municipality code (GMxxxx) instead, because both files contain this code. Joining all datasets on this code would be more robust, because it avoids spelling differences and makes municipal mergers visible.

Table 3 gives the definitions of all variables used in the analyses.

*Table 3. Definitions of the variables used in the analyses.*

| Variable | Definition | Unit |
| --- | --- | --- |
| **Charging point density** | Regular public charging points divided by the number of private households × 1,000 | charging points per 1,000 households |
| **Income** | Standardised household income, corrected for differences in household size. Both the mean and the median are reported; the median is less affected by a few very rich households | € 1,000 |
| **Age structure** | Share of residents aged 0–25 and share aged 65 and over, as a percentage of the total population | % |
| **Highly educated** | Persons aged 15–75 with hbo or wo as highest completed level, divided by all persons aged 15–75 with a known education level | % |
| **Left-wing vote share** | Votes for GroenLinks-PvdA, SP, Partij voor de Dieren, DENK, BIJ1, D66 and Volt, divided by all valid votes for party lists. A narrow definition without D66 and Volt is used as a robustness check. Postal votes from abroad are left out, because they do not belong to a municipality | % |
| **Population density** | Inhabitants per km² of land. Strongly right-skewed (from 23 on Schiermonnikoog to 6,868 in Den Haag, median 479), so its natural logarithm is used in the regression | inhabitants per km² |
| **Share of apartments** | Apartments as a percentage of all dwellings; a second indicator of the need for public charging | % |

## 5. Data pipeline

The data analysis consists of multiple notebooks. Each notebook reads its input from the raw data folder or from the processed data folder, and stores its output in figures or again in the processed data folder. The notebooks can be split into three main functions: data processing (0.1 to 0.6), exploration (1 to 5) and analysis (6 to 8).

1. **Collection.** Six datasets were downloaded and are stored in data/raw/, see Table 2.
2. **Extraction.** Notebook 0.5 uses the kwb2024 zip, extracts only the variables on education, population density and share of apartments, and saves them as kwb2024_per_gemeente.csv. It is stored in data/raw/ because it is input for several other notebooks.
3. **Data processing** (notebooks 0.1 to 0.6, and notebook 1). Each notebook prepares one variable from the raw data. Text values are converted to numbers, commas are replaced by points, total rows are removed, and municipality names are aligned with their spelling in the charging point file. Each notebook saves the processed data file for its variable in data/processed/. Notebook 1 combines the number of charging points, the number of households and the average income per municipality.
4. **Exploration per variable** (notebooks 1.2, 2, 2.2, 3, 4 and 5). These notebooks only read processed files and make a map per municipality; they do not save new data. Notebook 2 calculates the dependent variable, charging points per 1,000 households, from chargingpoints_households_income.csv. Notebook 1.2 maps income from the file of notebook 0.6, and notebooks 2.2, 3, 4 and 5 map population density, age structure, left-wing vote share and education level from the files of notebooks 0.4, 0.1, 0.3 and 0.2. All exploration notebooks use chargingpoints_households_income.csv to check that the municipality names match.
5. **Merging** (notebook 6). Notebook 6 joins all variables into one table with one row per municipality (all_variables_per_municipality.csv). It reads the processed files of notebooks 0.1–0.4 for age structure, left-wing vote share, education level and population density. Charging points, households and income are read from the raw files, and the share of apartments from the CBS extract, because no processed file contains these variables in the required form. This table is the input for all further analyses.
6. **Descriptive analysis** (notebooks 6 and 6.2). Notebook 6 uses the merged table directly, and notebook 6.2 reads it from all_variables_per_municipality.csv. Notebook 6 saves the correlation of each factor with charging point density in table_comparison_factors.csv, and the histogram and correlation matrix in figures/. Notebook 6.2 saves the boxplot and scatter plots in figures/. These outputs are not used by other notebooks.
7. **Regression** (notebook 7). Notebook 7 reads all_variables_per_municipality.csv. Inside the notebook, it gives the variables short names, takes the logarithm of population density, and converts all variables to z-scores. The regression results are saved in regression_results.csv, and the coefficient and residual plots in figures/.
8. **Stepwise regression** (notebook 8). Notebook 8 reads the same table, all_variables_per_municipality.csv, and prepares the variables in the same way as notebook 7. Its results are saved in three files in data/processed/ (stepwise_results.csv, all_subsets_results.csv and composite_model_results.csv), and two figures are saved in figures/.